## 1. Customer Loss Ratios
- Task:
-Given a list of insurance records with fields: customer_id, premium, and claim_amount, write a function that:
•	Aggregates by customer_id
•	Computes:
o	total_premium
o	total_claims
o	loss_ratio = total_claims / total_premium (handle divide-by-zero)
•	Returns a list of customers sorted by descending loss_ratio, then by customer_id.
What it tests:
Dictionaries, grouping/aggregation, sorting with custom keys, defensive coding.
________________________________________

In [9]:
# Test dataset: list of dicts
records1 = [
    # Customer 101: moderate loss ratio (claims < premium)
    {"customer_id": 101, "premium": 1000.0, "claim_amount": 200.0},
    {"customer_id": 101, "premium": 500.0,  "claim_amount": 100.0},

    # Customer 202: high loss ratio (claims > premium)
    {"customer_id": 202, "premium": 800.0,  "claim_amount": 900.0},
    {"customer_id": 202, "premium": 200.0,  "claim_amount": 400.0},

    # Customer 303: no claims (loss ratio = 0)
    {"customer_id": 303, "premium": 1200.0, "claim_amount": 0.0},
    {"customer_id": 303, "premium": 300.0,  "claim_amount": 0.0},

    # Customer 404: zero premium but has claims (tests divide-by-zero handling)
    {"customer_id": 404, "premium": 0.0,    "claim_amount": 500.0},

    # Customer 505: same loss ratio as 101 to test secondary sort by customer_id
    {"customer_id": 505, "premium": 1500.0, "claim_amount": 300.0},
]


In [11]:
records1

[{'customer_id': 101, 'premium': 1000.0, 'claim_amount': 200.0},
 {'customer_id': 101, 'premium': 500.0, 'claim_amount': 100.0},
 {'customer_id': 202, 'premium': 800.0, 'claim_amount': 900.0},
 {'customer_id': 202, 'premium': 200.0, 'claim_amount': 400.0},
 {'customer_id': 303, 'premium': 1200.0, 'claim_amount': 0.0},
 {'customer_id': 303, 'premium': 300.0, 'claim_amount': 0.0},
 {'customer_id': 404, 'premium': 0.0, 'claim_amount': 500.0},
 {'customer_id': 505, 'premium': 1500.0, 'claim_amount': 300.0}]

In [24]:
def compute_customer_loss_ratio(records):
    #create empyty dictionary to store records per customer
    aggregates = {}

    for record in records:
        customer_id = record['customer_id']
        premium = record['premium']
        claim = record['claim_amount']


        if customer_id not in aggregates:
            aggregates[customer_id] = {
                'total_premium':0.0,
                'total_claim':0.0    
            }
        #add this record to the customer new aggregate dict
        aggregates[customer_id]['total_premium'] += premium
        aggregates[customer_id]['total_claim'] += claim

    result = []

    for customer_id, totals in aggregates.items():
        total_premium = totals['total_premium']
        total_claim = totals['total_claim']

        #cal loss ratio
        if total_premium == 0:
            loss_ratio = 0
        else:
            loss_ratio = total_claim/total_premium
        result.append({
        'customer_id':customer_id,
        'total_premium':  total_premium,
        'total_claim' : total_claim,
        'loss ratio' : loss_ratio,
    })   
    result_sorted = sorted(result, key = lambda x:(-x['total_claim'], x[ 'customer_id']))
        
    return result_sorted


In [25]:
compute_customer_loss_ratio(records1)

[{'customer_id': 202,
  'total_premium': 1000.0,
  'total_claim': 1300.0,
  'loss ratio': 1.3},
 {'customer_id': 404,
  'total_premium': 0.0,
  'total_claim': 500.0,
  'loss ratio': 0},
 {'customer_id': 101,
  'total_premium': 1500.0,
  'total_claim': 300.0,
  'loss ratio': 0.2},
 {'customer_id': 505,
  'total_premium': 1500.0,
  'total_claim': 300.0,
  'loss ratio': 0.2},
 {'customer_id': 303,
  'total_premium': 1500.0,
  'total_claim': 0.0,
  'loss ratio': 0.0}]

## 2. Fraud Rule Engine (Boolean Logic)
Task:
You’re given a list of transactions, each as a dict with keys:
amount, country, is_international, channel (e.g. "online", "pos", "atm").
Write a function that returns the number of suspicious transactions where:
•	amount > 5000, or
•	is_international == True and amount > 1000, or
•	channel == "online" and country not in an allowed list.
What it tests:
Control flow, boolean logic, list/dict handling, clear code structure.


In [26]:
#data
transactions = [
    # 1) Suspicious by rule1: amount > 5000
    {
        "amount": 6000.0,
        "country": "UK",
        "is_international": False,
        "channel": "pos",
    },
    # 2) Suspicious by rule2: is_international == True and amount > 1000
    {
        "amount": 1500.0,
        "country": "France",
        "is_international": True,
        "channel": "pos",
    },
    # 3) Suspicious by rule3: channel == "online" and country NOT in allowed list
    {
        "amount": 800.0,
        "country": "Brazil",
        "is_international": True,
        "channel": "online",
    },
    # 4) Not suspicious: small amount, online, but country is allowed
    {
        "amount": 200.0,
        "country": "USA",
        "is_international": False,
        "channel": "online",
    },
    # 5) Suspicious: triggers rule1 and rule2 (still counts as 1)
    {
        "amount": 5200.0,
        "country": "Germany",
        "is_international": True,
        "channel": "online",
    },
    # 6) Not suspicious: low amount, domestic, atm
    {
        "amount": 900.0,
        "country": "India",
        "is_international": False,
        "channel": "atm",
    },
    # 7) Suspicious by rule3: online + country not allowed
    {
        "amount": 1200.0,
        "country": "Spain",
        "is_international": False,
        "channel": "online",
    },
    # 8) Suspicious by rule2: international + amount > 1000
    {
        "amount": 3000.0,
        "country": "USA",
        "is_international": True,
        "channel": "pos",
    },
]


In [26]:
#data
transactions = [
    # 1) Suspicious by rule1: amount > 5000
    {
        "amount": 6000.0,
        "country": "UK",
        "is_international": False,
        "channel": "pos",
    },
    # 2) Suspicious by rule2: is_international == True and amount > 1000
    {
        "amount": 1500.0,
        "country": "France",
        "is_international": True,
        "channel": "pos",
    },
    # 3) Suspicious by rule3: channel == "online" and country NOT in allowed list
    {
        "amount": 800.0,
        "country": "Brazil",
        "is_international": True,
        "channel": "online",
    },
    # 4) Not suspicious: small amount, online, but country is allowed
    {
        "amount": 200.0,
        "country": "USA",
        "is_international": False,
        "channel": "online",
    },
    # 5) Suspicious: triggers rule1 and rule2 (still counts as 1)
    {
        "amount": 5200.0,
        "country": "Germany",
        "is_international": True,
        "channel": "online",
    },
    # 6) Not suspicious: low amount, domestic, atm
    {
        "amount": 900.0,
        "country": "India",
        "is_international": False,
        "channel": "atm",
    },
    # 7) Suspicious by rule3: online + country not allowed
    {
        "amount": 1200.0,
        "country": "Spain",
        "is_international": False,
        "channel": "online",
    },
    # 8) Suspicious by rule2: international + amount > 1000
    {
        "amount": 3000.0,
        "country": "USA",
        "is_international": True,
        "channel": "pos",
    },
]


In [32]:
allowed_countries_for_online = {"UK", "USA", "Germany"}


In [28]:
transactions

[{'amount': 6000.0,
  'country': 'UK',
  'is_international': False,
  'channel': 'pos'},
 {'amount': 1500.0,
  'country': 'France',
  'is_international': True,
  'channel': 'pos'},
 {'amount': 800.0,
  'country': 'Brazil',
  'is_international': True,
  'channel': 'online'},
 {'amount': 200.0,
  'country': 'USA',
  'is_international': False,
  'channel': 'online'},
 {'amount': 5200.0,
  'country': 'Germany',
  'is_international': True,
  'channel': 'online'},
 {'amount': 900.0,
  'country': 'India',
  'is_international': False,
  'channel': 'atm'},
 {'amount': 1200.0,
  'country': 'Spain',
  'is_international': False,
  'channel': 'online'},
 {'amount': 3000.0,
  'country': 'USA',
  'is_international': True,
  'channel': 'pos'}]

In [38]:
for tx in transactions:
    amount = tx['amount']
    print(amount) 

6000.0
1500.0
800.0
200.0
5200.0
900.0
1200.0
3000.0


In [52]:
def fraud_tran_detec_tran_count(all_tran_records, country_list):
    """ Function to detect suspicious transactions
    
    parameters
    
    
    """
    country_list = set(country_list)
    sus_tran_rec = []

    sus_tran = 0
    # create a variables to store each feature values
    for record in all_tran_records:
        amount = record['amount']
        country = record['country']
        is_international = record['is_international']
        channel = record['channel']

    #create conditions
        rule_1 = amount > 5000 
        rule_2 = is_international ==True and amount > 1000
        rule_3 = channel == 'online' and country not in country_list

        #check condtions and count then return susp transactipns
        if rule_1 or rule_2 or rule_3:
            sus_tran += 1
            sus_tran_rec.append(record)
    record_summary = (sus_tran_rec, sus_tran)       
            
    return record_summary 
    
    

In [53]:
fraud_tran_detec_tran_count(transactions,allowed_countries_for_online)

([{'amount': 6000.0,
   'country': 'UK',
   'is_international': False,
   'channel': 'pos'},
  {'amount': 1500.0,
   'country': 'France',
   'is_international': True,
   'channel': 'pos'},
  {'amount': 800.0,
   'country': 'Brazil',
   'is_international': True,
   'channel': 'online'},
  {'amount': 5200.0,
   'country': 'Germany',
   'is_international': True,
   'channel': 'online'},
  {'amount': 1200.0,
   'country': 'Spain',
   'is_international': False,
   'channel': 'online'},
  {'amount': 3000.0,
   'country': 'USA',
   'is_international': True,
   'channel': 'pos'}],
 6)

## RE-WRITE
Task: You’re given a list of transactions, each as a dict with keys: amount, country, is_international, channel (e.g. "online", "pos", "atm"). Write a function that returns the number of suspicious transactions where: • amount > 5000, or • is_international == True and amount > 1000, or • channel == "online" and country not in an allowed list. What it tests: Control flow, boolean logic, list/dict handling, clear code structure.

In [41]:
def fraud_rule_engine(fraud_record, country_list):
    #convert the list of countries to a set 
    country_list = set(country_list)

    #create variable to store count of suspicious transactions
    suspicious_tran = 0

    
    
    #amount greater that 5000
    for record in fraud_record:
        amount = record['amount']
        country = record ['country']
        is_international= record ['is_international']
        channel = record['channel']
            


        #define rules
        rule_1 = amount > 5000
        rule_2 = is_international == True and amount > 1000
        rule_3 = (channel == 'online') and (country not in country_list)

        if rule_1 or rule_2 or rule_3:
            suspicious_tran +=1
            
    return suspicious_tran    
    

In [42]:
fraud_rule_engine(transactions,allowed_countries_for_online)

6

In [59]:
li = [2, 3, 4, 5, 6, 1]

df1 = pd.DataFrame( li, columns= ['Numbers'])
print(df1)


   Numbers
0        2
1        3
2        4
3        5
4        6
5        1


In [60]:
df3 = pd.DataFrame([li], columns = [f'col_{i}' for i in range(len(li))])
df3

,col_0,col_1,col_2,col_3,col_4,col_5
0,2,3,4,5,6,1


In [56]:
import pandas as pd
df = pd.DataFrame([li], columns=[f"col_{i}" for i in range(len(li))])
print(df)


   col_0  col_1  col_2  col_3  col_4  col_5
0      2      3      4      5      6      1


## 3. Moving Average of Claim Amounts
* Task:
- Given a list of daily claim amounts [a1, a2, ..., an] and a window size k, return a list of k-day moving averages (as floats rounded to 2 decimals).
- What it tests:
   Loops, slicing, indices, complexity awareness (O(nk) vs O(n)), basic numerics.


## 4. Policy Reference Validator (Strings / Regex)
Task:
Given a list of policy references (strings), mark each one as valid or invalid. A valid reference:
•	Has format: 3 uppercase letters, a dash, then 7 digits (e.g., "HIS-1234567").
Return a list of booleans or the count of valid ones.
What it tests:
String manipulation, regex (optional but nice), edge cases.
________________________________________


## 5. Removing Duplicate Policies (Sets & Dicts)
Task:
You receive a list of policy records (dictionaries) where some are duplicates. The combination (policy_id, effective_date) defines uniqueness.
Write a function to remove duplicates while preserving the original order of first appearance.
What it tests:
Sets, tuples as keys, stable de-duplication logic.


## 6. Basic Statistics Helper
Task:
Implement a function that takes a list of numbers and returns:
•	mean
•	median
•	population standard deviation
Do not use numpy or statistics – implement the formulas manually.
What it tests:
Math correctness, loops, sorting, careful handling of odd/even lengths, numerical reasoning (very data-sciency).
________________________________________


## 7. Confusion Matrix & F1 Score
Task:
Given two equal-length lists:
•	y_true – actual labels (0 or 1)
•	y_pred – predicted labels (0 or 1)
Compute:
•	TP, FP, TN, FN
•	precision, recall, and F1 score (handle edge cases: zero division)
Return them as a dictionary.
What it tests:
Understanding of classification metrics (important for fraud / risk models), clean Python implementation.


## 8. Group Claims by Month (Dates / Parsing)
Task:
You’re given a list of claim records with claim_date as "YYYY-MM-DD" string and amount as float.
Write a function that returns a dictionary mapping "YYYY-MM" → total_claim_amount for that month.
What it tests:
String slicing or datetime parsing, grouping by derived key, basic aggregations.
________________________________________


## 9. Max Overlapping Claims (Intervals Problem)
Task:
Each claim has a start and end day (integers, e.g. day of year). You get a list of (start_day, end_day) intervals.
Return the maximum number of claims open at the same time.
What it tests:
Classic algorithm problem (sweep line / event points), sorting, reasoning about intervals.


## 10. Inner Join in Pure Python
- Task:
You have:
•	policies: list of dicts with policy_id, customer_id
•	customers: list of dicts with customer_id, name, country
Implement an inner join on customer_id and return a list of joined dicts:

- {
  "policy_id": ...,
  "customer_id": ...,
  "name": ...,
  "country": ...
}

- nWhat it tests:
Hash maps (dict for fast lookup), join logic, dealing with missing keys.


## SImple Task 


#### 1. Task: Write a function that returns the sum of a list of numbers.
#### Test
print(sum_list([1, 2, 3, 4]))  # 10


In [65]:
def sum_list(li):
    final_val = 0
    for i in li:
        final_val +=i
    return final_val    

In [66]:
mi = [1, 2, 3, 4]
sum_list(mi)

10

#### 2. Task: 3. Remove Duplicates While Preserving Order
- Task: Given a list, remove duplicates but keep the original order of first occurrence.

#### Test
print(dedupe_preserve_order([1, 2, 2, 3, 1]))  # [1, 2, 3]


In [69]:
def dedupe_pre_order(li):
    seen = set()
    final_li = []
    for x in li:
        if x not in seen:
            seen.add(x)
            final_li.append(x)
    return final_li
    
            
        

In [71]:
v = [1, 2, 2, 3, 1]
v2 = [3, 1, 3, 2, 1]
dedupe_pre_order(v)
dedupe_pre_order(v2)

[3, 1, 2]

### Task 4
- Word Frequency Counter
Task: Given a string, return a dict of word → count.


Test
print(word_counts("This is a test this is"))  


In [75]:
def word_counter(sentence):
    dic_count = {}
    words = sentence.lower().split()
    for word in words:
        if word not in dic_count:
            dic_count[word] = 0
        dic_count[word] +=1
    return dic_count
    
        
    

In [76]:
word_counter("This is a test this is")

{'this': 2, 'is': 2, 'a': 1, 'test': 1}

### Task 5: Implement basic statistics for a list of numbers.


- Mean, Median, Standard Deviation (No Libraries)
-  Test  print(basic_stats([1, 2, 3, 4, 5]))


In [85]:
import math

def basic_stats(num_list):
    #mean
    n = len(num_list)
    if n == 0:
        raise ValueError ("List value is not correct")
        
    mean = sum(num_list)/n
    
    #median
    sorted_list = sorted(num_list)
    mid = n // 2
    if n % 2 == 1:
        median = sorted_list[mid]
    else:
        median = sum(sorted_list[mid -1] + sorted_list[mid])/2

    #standard deviation
    var = sum((x-mean) **2 for x in num_list)/n
    std =math.sqrt(var)   

    return mean, median, std
    

In [86]:
basic_stats([1, 2, 3, 4, 5])

(3.0, 3, 1.4142135623730951)

In [81]:
import math

def basic_stats2(nums):
    n = len(nums)
    if n == 0:
        raise ValueError("Empty list")
    # mean
    mean = sum(nums) / n
    # median
    sorted_nums = sorted(nums)
    mid = n // 2
    if n % 2 == 1:
        median = sorted_nums[mid]
    else:
        median = (sorted_nums[mid - 1] + sorted_nums[mid]) / 2
    # population std
    var = sum((x - mean) ** 2 for x in nums) / n
    std = math.sqrt(var)
    return mean, median, std


In [82]:
basic_stats2([1, 2, 3, 4, 5])

(3.0, 3, 1.4142135623730951)

### 6. Min–Max Normalize a List to [0, 1]
-Task: Scale a list of numbers to the range [0, 1].


In [108]:
#solution 1
#normalize
def normaliza_list(lis_val):
    new_norm_list = []
    max_val = max(lis_val)
    min_val = min(lis_val)
    for x in lis_val:
        if x - min_val ==0 or max_val == min_val :
            v = 0
            new_norm_list.append(v)
        else:
            v = (x - min_val) / (max_val - min_val)
            new_norm_list.append(v)
    
    return new_norm_list    

In [1]:
#solution 2
def norm_val(list_val):
    max_val = max(list_val)
    min_val = min(list_val)
    if max_val == min_val:
        return [0.0] * len(list_val)
    else:
        return [(x - min_val)/(max_val - min_val) for x in list_val]
        


In [2]:
data1 = [10, 20, 30,23,50,200, 35, 14,56]


In [115]:
normaliza_list(data1)

[0,
 0.05263157894736842,
 0.10526315789473684,
 0.06842105263157895,
 0.21052631578947367,
 1.0,
 0.13157894736842105,
 0.021052631578947368,
 0.24210526315789474]

In [3]:
norm_val(data1)

[0.0,
 0.05263157894736842,
 0.10526315789473684,
 0.06842105263157895,
 0.21052631578947367,
 1.0,
 0.13157894736842105,
 0.021052631578947368,
 0.24210526315789474]

In [96]:
def min_max_scale(nums):
    min_val = min(nums)
    max_val = max(nums)
    if max_val == min_val:
        return [0.0] * len(nums)
    return [(x - min_val) / (max_val - min_val) for x in nums]




In [101]:
# Test

min_max_scale(data1)
  # [0.0, 0.5, 1.0]
#Explanation:
#Apply (x - min) / (max - min) to each element. Handle the edge case when all values are equal.

[0.0,
 0.05263157894736842,
 0.10526315789473684,
 0.06842105263157895,
 0.21052631578947367,
 1.0,
 0.13157894736842105,
 0.021052631578947368,
 0.24210526315789474]

### Quick Questions on Missing values


In [117]:
#Question:You are given a pandas DataFrame df. Write code to return the number of missing values in each column.
import pandas as pd
def missin_val_per_column(df: pd.DataFrame)-> pd.Series:
    return df.isnull().sum()

In [162]:
df = pd.read_csv("financial_data_1000_records.csv")
df.head(5)

,customer_id,full_name,age,country,account_open_date,transaction_date,transaction_type,transaction_amount,account_balance,is_international
0,221958,Jordan Smith,34,India,2018-04-14,2024-03-11,ATM,1584.01,17857.01,False
1,771155,Casey Williams,22,France,2019-06-14,2024-04-27,Online,-463.09,13702.98,False
2,231932,Morgan Brown,46,Germany,2017-07-02,2023-01-25,POS,1622.31,20970.86,False
3,465838,Jordan Anderson,21,India,2016-03-29,2024-10-14,ATM,3854.46,20404.84,False
4,359178,Jordan Thomas,27,Italy,2015-02-06,2023-04-27,Online,2777.94,20849.51,False


In [119]:
missin_val_per_column(df)

customer_id           0
full_name             0
age                   0
country               0
account_open_date     0
transaction_date      0
transaction_type      0
transaction_amount    0
account_balance       0
is_international      0
dtype: int64

In [163]:
df_train = df.drop(columns = ['is_international'])
df_train.head(5)

,customer_id,full_name,age,country,account_open_date,transaction_date,transaction_type,transaction_amount,account_balance
0,221958,Jordan Smith,34,India,2018-04-14,2024-03-11,ATM,1584.01,17857.01
1,771155,Casey Williams,22,France,2019-06-14,2024-04-27,Online,-463.09,13702.98
2,231932,Morgan Brown,46,Germany,2017-07-02,2023-01-25,POS,1622.31,20970.86
3,465838,Jordan Anderson,21,India,2016-03-29,2024-10-14,ATM,3854.46,20404.84
4,359178,Jordan Thomas,27,Italy,2015-02-06,2023-04-27,Online,2777.94,20849.51


In [122]:
import pandas as pd
import numpy as np

# Sample DataFrame
df_miss = pd.DataFrame({
    "customer_id": [101, 102, 103, 104, 105],
    "age":         [25, 32, np.nan, 45, 29],
    "country":     ["UK", "USA", "France", None, "Germany"],
    "income":      [40000, np.nan, 52000, 61000, None],
    "is_active":   [True, True, False, True, False]
})

print(df_miss)


   customer_id   age  country   income  is_active
0          101  25.0       UK  40000.0       True
1          102  32.0      USA      NaN       True
2          103   NaN   France  52000.0      False
3          104  45.0     None  61000.0       True
4          105  29.0  Germany      NaN      False


In [124]:
cols = ["age", "country"]

In [152]:
#Q2: Question:Given a DataFrame df and a list of column names cols, 
# write a functions that drops all columns where any of those columns has a missing value, and returns the cleaned DataFrame.
#write a functions that drops all rows where any of those columns has a missing value, and returns the cleaned DataFrame.
#

def drop_colum_clean(df):
    return df.dropna(axis = 1)
    
   
            
            
            
        
    


In [153]:
drop_colum_clean(df_miss)

,customer_id,is_active
0,101,True
1,102,True
2,103,False
3,104,True
4,105,False


In [156]:
def drop_row_clean(df,cols):
    missing_cols = [col for col in cols if col not in df.columns]
    if missing_cols:
        raise KeyError(f"column not found: {missing_cols}")
    return df.dropna(axis=0, subset = cols)    

In [157]:
drop_row_clean(df_miss,cols)

,customer_id,age,country,income,is_active
0,101,25.0,UK,40000.0,True
1,102,32.0,USA,NaN,True
4,105,29.0,Germany,NaN,False


In [146]:
def drop_rows_with_missing(df, cols):
    missing_cols = [c for c in cols if c not in df.columns]
    if missing_cols:
        raise KeyError(f"Columns not in DataFrame: {missing_cols}")
    return df.dropna(subset=cols)

df_clean = drop_rows_with_missing(df_miss, cols)
print(df_clean)


   customer_id   age  country   income  is_active
0          101  25.0       UK  40000.0       True
1          102  32.0      USA      NaN       True
4          105  29.0  Germany      NaN      False


### missing value Task 3: Fill missing numeric values with the column mean
Question:
Write a function that fills missing values in all numeric columns of a DataFrame with the mean of that column. Return the updated DataFrame (do not modify the original).


In [158]:
import pandas as pd
import numpy as np

# Sample DataFrame with numeric and non-numeric columns
dfmr = pd.DataFrame({
    "customer_id": [101, 102, 103, 104, 105],
    "age":         [25, 32, np.nan, 45, 29],
    "income":      [40000, np.nan, 52000, 61000, np.nan],
    "country":     ["UK", "USA", "France", "Germany", "UK"],
    "score":       [3.5, np.nan, 4.2, 4.8, 3.9],
})

print(dfmr)


   customer_id   age   income  country  score
0          101  25.0  40000.0       UK    3.5
1          102  32.0      NaN      USA    NaN
2          103   NaN  52000.0   France    4.2
3          104  45.0  61000.0  Germany    4.8
4          105  29.0      NaN       UK    3.9


In [ ]:
def missin_replace_mean(df):
    

### Task 7
- Dot Product of Two Vectors (NumPy)
- Task: Compute the dot product of two equal-length lists using NumPy.


In [1]:
x,y,z = 1,2,3
li = []
li.append(x)
li

[1]

In [19]:
dic = {'name': 'kola',
       'age':12,
       'salary':45000,
       'postcode':'SE23s0',
      }

In [20]:
dic

{'name': 'kola', 'age': 12, 'salary': 45000, 'postcode': 'SE23s0'}

In [21]:
dic['name'] = 'wole'

In [22]:
dic

{'name': 'wole', 'age': 12, 'salary': 45000, 'postcode': 'SE23s0'}

In [ ]:
for item, val in dic.items:
    name_id = val['name']
    age_id = val['age']
    
    